# U23AI035 - LAB 11 - IMAES
## Experiment 11: Medical Diagnosis with Certainty Factors

**SVNIT, Surat | Department of Artificial Intelligence | Semester VII**

**Aim:** Build a diagnostic expert system that uses certainty factors (CFs), ranks four candidate diagnoses, accounts for uncertain patient-reported findings, and selects only the questions needed to reach a conclusion.

> **Academic-use note:** This is a small illustrative expert-system model for demonstrating certainty-factor reasoning. Its rules and CF values are invented for the lab and are not clinically validated. It must not be used to diagnose or treat real patients.

## 1. Theory and design choices

A certainty factor is a number in [-1, +1]. Positive values support a finding or hypothesis, negative values oppose it, and zero means unknown.

The implementation uses the three operations described in the experiment:
1. **Conjunction strength:** minimum of the strengths of all required findings.
2. **Single-rule contribution:** rule CF × conjunction strength.
3. **Parallel evidence:** combine contributions supporting the same diagnosis. For same-sign values, use the standard CF combination formula; for opposite signs, use the standard conflict formula.

For a rule condition that requires a finding to be **present**, its strength is the patient's CF for that finding. For a condition that requires a finding to be **absent**, its strength is the negative of the patient's CF. Thus, a denial (negative patient CF) supports an absent condition, while an unknown value (zero) provides no support.

The four fictional candidate diagnoses are **Common Cold, Influenza-like Illness, Allergic Rhinitis,** and **Sinusitis**. The conclusion threshold is set to **0.60** for this demonstration.

## 2. Rule base (eight rules over four candidate diagnoses)

In [ ]:
# Each rule is:
# diagnosis, required_present findings, required_absent findings, rule certainty factor
RULES = [
    {"id":"R1", "diagnosis":"Common Cold", "present":["runny_nose","sore_throat"], "absent":[], "cf":0.70},
    {"id":"R2", "diagnosis":"Common Cold", "present":["cough","sneezing"], "absent":["high_fever"], "cf":0.65},
    {"id":"R3", "diagnosis":"Influenza-like Illness", "present":["high_fever","body_ache"], "absent":[], "cf":0.85},
    {"id":"R4", "diagnosis":"Influenza-like Illness", "present":["fatigue","chills"], "absent":["sneezing"], "cf":0.70},
    {"id":"R5", "diagnosis":"Allergic Rhinitis", "present":["sneezing","itchy_eyes"], "absent":["high_fever"], "cf":0.85},
    {"id":"R6", "diagnosis":"Allergic Rhinitis", "present":["runny_nose","itchy_eyes"], "absent":["body_ache"], "cf":0.75},
    {"id":"R7", "diagnosis":"Sinusitis", "present":["facial_pain","nasal_congestion"], "absent":[], "cf":0.80},
    {"id":"R8", "diagnosis":"Sinusitis", "present":["nasal_congestion","headache"], "absent":["sneezing"], "cf":0.65},
]

FINDINGS = sorted({f for r in RULES for f in r["present"] + r["absent"]})
DIAGNOSES = sorted({r["diagnosis"] for r in RULES})

rule_table = pd.DataFrame([
    {"Rule":r["id"], "Diagnosis":r["diagnosis"],
     "Required present":", ".join(r["present"]) or "—",
     "Required absent":", ".join(r["absent"]) or "—", "Rule CF":r["cf"]}
    for r in RULES
])
display(rule_table)
print("Findings:", FINDINGS)

## 3. Certainty-factor operations and inference engine

In [ ]:
def combine_two_cf(cf1, cf2):
    """Combine two certainty factors using standard CF combination cases."""
    if cf1 >= 0 and cf2 >= 0:
        return cf1 + cf2 - cf1 * cf2
    if cf1 <= 0 and cf2 <= 0:
        return cf1 + cf2 + cf1 * cf2
    denominator = 1 - min(abs(cf1), abs(cf2))
    if denominator == 0:
        return 0.0
    return (cf1 + cf2) / denominator

def combine_many_cfs(values):
    values = list(values)
    if not values:
        return 0.0
    result = values[0]
    for value in values[1:]:
        result = combine_two_cf(result, value)
    return float(np.clip(result, -1.0, 1.0))

def rule_antecedent_strength(rule, patient):
    strengths = [patient.get(f, 0.0) for f in rule["present"]]
    strengths += [-patient.get(f, 0.0) for f in rule["absent"]]
    if not strengths:
        return 0.0
    return min(strengths)

def evaluate_patient(patient, rules=RULES):
    contributions = {d: [] for d in DIAGNOSES}
    details = []
    for rule in rules:
        strength = rule_antecedent_strength(rule, patient)
        contribution = rule["cf"] * strength
        contributions[rule["diagnosis"]].append(contribution)
        details.append({
            "Rule":rule["id"], "Diagnosis":rule["diagnosis"],
            "Antecedent strength":strength,
            "Rule CF":rule["cf"], "Contribution":contribution
        })
    combined = {d: combine_many_cfs(vals) for d, vals in contributions.items()}
    ranked = sorted(combined.items(), key=lambda item:item[1], reverse=True)
    return ranked, details, contributions

def print_differential(patient, title="Patient"):
    ranked, details, contributions = evaluate_patient(patient)
    print(f"\n{title}")
    print("Patient findings (CF):", patient)
    print("\nRanked differential:")
    for rank, (diagnosis, cf) in enumerate(ranked, 1):
        contributing = [
            f"{r['id']} ({r['Contribution']:+.3f})"
            for r in details if r["Diagnosis"] == diagnosis and abs(r["Contribution"]) > 1e-12
        ]
        print(f"{rank}. {diagnosis}: combined CF = {cf:.3f}; rules: {', '.join(contributing) or 'none'}")
    leader, leader_cf = ranked[0]
    if leader_cf >= 0.60:
        print(f"Conclusion: {leader} (threshold passed: {leader_cf:.3f} ≥ 0.60)")
    else:
        print(f"Conclusion: No diagnosis reaches the threshold (leader: {leader}, CF={leader_cf:.3f})")
    return ranked, pd.DataFrame(details)

## 4. Evaluate a patient and show rule contributions

In [ ]:
# Values represent confidence in the presence of each finding:
# +1 = definitely present, -1 = definitely absent, 0 = unknown.
DEMO_PATIENT = {
    "runny_nose":0.90, "sore_throat":0.80, "cough":0.70,
    "sneezing":0.10, "high_fever":-0.80, "body_ache":-0.50,
    "fatigue":0.20, "chills":-0.30, "itchy_eyes":0.90,
    "facial_pain":-0.70, "nasal_congestion":0.60, "headache":0.20
}
ranked, contribution_table = print_differential(DEMO_PATIENT, "Demonstration patient")
display(contribution_table.style.format({
    "Antecedent strength":"{:.3f}", "Rule CF":"{:.2f}", "Contribution":"{:+.3f}"
}))

## 5. Driving the consultation: ask one finding at a time

The consultation starts with every finding unknown (CF = 0). Each strategy chooses one unanswered finding, receives the patient's stored response for that finding, and recalculates the ranked differential. It stops as soon as the leading diagnosis reaches the threshold or no relevant question remains.

The three strategies are:
- **Fixed intake order:** asks findings in a predetermined form order.
- **Frequency order:** asks findings in descending order of how often they occur in the rule base.
- **Goal-directed:** after each answer, focuses on findings used by rules for the currently leading diagnosis; if there is no leader yet, it uses the frequency order to obtain initial evidence.

In [ ]:
INTAKE_ORDER = [
    "high_fever","cough","runny_nose","sore_throat","sneezing","body_ache",
    "fatigue","chills","itchy_eyes","facial_pain","nasal_congestion","headache"
]

from collections import Counter
finding_frequency = Counter(
    f for rule in RULES for f in (rule["present"] + rule["absent"])
)
FREQUENCY_ORDER = sorted(FINDINGS, key=lambda f:(-finding_frequency[f], f))

def current_leader(patient):
    ranked, _, _ = evaluate_patient(patient)
    return ranked[0]

def goal_directed_candidates(leader):
    return sorted({
        f for rule in RULES if rule["diagnosis"] == leader
        for f in (rule["present"] + rule["absent"])
    }, key=lambda f:(-finding_frequency[f], f))

def run_consultation(true_patient, strategy, threshold=0.60):
    known = {f:0.0 for f in FINDINGS}
    asked = []
    remaining = set(FINDINGS)

    while remaining:
        ranked, _, _ = evaluate_patient(known)
        leader, leader_cf = ranked[0]
        if leader_cf >= threshold:
            return {"strategy":strategy, "questions":asked,
                    "conclusion":leader, "confidence":leader_cf,
                    "threshold_passed":True, "final_ranked":ranked}

        if strategy == "Fixed intake":
            candidates = [f for f in INTAKE_ORDER if f in remaining]
        elif strategy == "Frequency":
            candidates = [f for f in FREQUENCY_ORDER if f in remaining]
        elif strategy == "Goal-directed":
            relevant = [f for f in goal_directed_candidates(leader) if f in remaining]
            candidates = relevant or [f for f in FREQUENCY_ORDER if f in remaining]
        else:
            raise ValueError("Unknown strategy")

        question = candidates[0]
        asked.append(question)
        known[question] = true_patient.get(question, 0.0)
        remaining.remove(question)

    ranked, _, _ = evaluate_patient(known)
    leader, leader_cf = ranked[0]
    passed = leader_cf >= threshold
    return {"strategy":strategy, "questions":asked,
            "conclusion":leader if passed else "No diagnosis reaches threshold",
            "confidence":leader_cf, "threshold_passed":passed,
            "final_ranked":ranked}

consultation_results = [
    run_consultation(DEMO_PATIENT, strategy)
    for strategy in ["Fixed intake", "Frequency", "Goal-directed"]
]
for result in consultation_results:
    print("\nStrategy:", result["strategy"])
    print("Questions asked ({}):".format(len(result["questions"])), " → ".join(result["questions"]))
    print("Conclusion:", result["conclusion"])
    print("Leading CF:", f"{result['confidence']:.3f}")
    print("Threshold passed:", result["threshold_passed"])

## 6. Observations and comments

1. **Combined CF versus either rule alone:** When two rules provide positive support for the same diagnosis, the combined CF is greater than either individual contribution (unless one contribution is zero or already at the limit). The combination accumulates evidence but remains bounded by +1.
2. **Effect of a weakly reported finding in a conjunction:** The minimum operation makes the conjunction no stronger than its weakest required finding. Therefore, even a rule with a high rule CF contributes little if one required symptom is reported with low confidence.
3. **Denial versus unknown:** A denial is represented by a negative CF and can actively support rules that require a finding to be absent. Unknown is represented by zero and contributes no positive support to either presence or absence.
4. **When no diagnosis passes the threshold:** The system should report that no candidate has sufficient certainty, show the ranked differential and CF values, and recommend gathering more information or human review. It should not force a diagnosis.

## 7. Exercise 1 — Sensitivity to changes in rule certainty factors

Generate a population of synthetic patients. For each patient, perturb every rule CF independently by a uniform random amount within ±0.05, ±0.10 and ±0.20, clip the result to [-1, +1], and compare the leading diagnosis and leading CF with the unperturbed model. Also count cases where a baseline consultation passed the threshold but the perturbed consultation no longer does.

In [ ]:
def generate_patient_population(n=300, seed=111):
    rng = np.random.default_rng(seed)
    population = []
    for _ in range(n):
        patient = {}
        for finding in FINDINGS:
            # Mix positive, negative and unknown/weakly reported findings.
            patient[finding] = float(np.clip(rng.normal(0, 0.65), -1, 1))
        population.append(patient)
    return population

PATIENT_POPULATION = generate_patient_population()
PERTURBATION_LEVELS = [0.05, 0.10, 0.20]
rng = np.random.default_rng(2027)

def perturbed_rules(epsilon, rng):
    changed = []
    for rule in RULES:
        r = rule.copy()
        r["cf"] = float(np.clip(rule["cf"] + rng.uniform(-epsilon, epsilon), -1, 1))
        changed.append(r)
    return changed

def evaluate_with_rules(patient, rules):
    ranked, _, _ = evaluate_patient(patient, rules)
    return ranked

baseline_records = []
for patient in PATIENT_POPULATION:
    ranked, _, _ = evaluate_patient(patient)
    baseline_records.append({
        "leader":ranked[0][0], "cf":ranked[0][1],
        "passed":ranked[0][1] >= 0.60
    })

sensitivity_rows = []
for epsilon in PERTURBATION_LEVELS:
    changed_leader = 0
    cf_differences = []
    passed_then_failed = 0
    for patient, baseline in zip(PATIENT_POPULATION, baseline_records):
        rules_variant = perturbed_rules(epsilon, rng)
        variant_ranked = evaluate_with_rules(patient, rules_variant)
        variant_leader, variant_cf = variant_ranked[0]
        changed_leader += int(variant_leader != baseline["leader"])
        cf_differences.append(variant_cf - baseline["cf"])
        passed_then_failed += int(baseline["passed"] and variant_cf < 0.60)

    sensitivity_rows.append({
        "Perturbation ±":epsilon,
        "Leading diagnosis changed":changed_leader,
        "Change rate (%)":100*changed_leader/len(PATIENT_POPULATION),
        "Mean change in leading CF":float(np.mean(cf_differences)),
        "Baseline passed but perturbed failed":passed_then_failed
    })

sensitivity_table = pd.DataFrame(sensitivity_rows)
display(sensitivity_table.style.format({
    "Perturbation ±":"{:.2f}", "Change rate (%)":"{:.1f}%",
    "Mean change in leading CF":"{:+.4f}"
}))

### Exercise 1 — Interpretation

The table measures how sensitive the system is to modest changes in expert-assigned rule certainty. Larger perturbation ranges generally create more opportunity for rankings and threshold decisions to change, although the exact rates depend on the generated patient population and random seed. A changed leading diagnosis indicates ranking instability; a baseline-passed/perturbed-failed case indicates that the threshold conclusion was not robust to the perturbation. The mean change in leading CF can be positive or negative because perturbations are random.

## 8. Exercise 2 — Compare question strategies over the whole population

Run all three consultation strategies for every synthetic patient. Report mean and maximum question counts and how often the final conclusions differ. The same patient answers are used for all strategies, so differences arise from question order and stopping behaviour.

In [ ]:
strategy_names = ["Fixed intake", "Frequency", "Goal-directed"]
population_strategy_results = {name:[] for name in strategy_names}

for patient in PATIENT_POPULATION:
    for strategy in strategy_names:
        population_strategy_results[strategy].append(
            run_consultation(patient, strategy)
        )

strategy_summary = []
for strategy in strategy_names:
    results = population_strategy_results[strategy]
    counts = [len(r["questions"]) for r in results]
    conclusions = [r["conclusion"] for r in results]
    strategy_summary.append({
        "Strategy":strategy,
        "Mean questions":np.mean(counts),
        "Maximum questions":np.max(counts),
        "Patients reaching threshold":sum(r["threshold_passed"] for r in results),
        "Distinct conclusions":len(set(conclusions))
    })

strategy_summary_df = pd.DataFrame(strategy_summary)
display(strategy_summary_df.style.format({
    "Mean questions":"{:.2f}", "Maximum questions":"{:.0f}"
}))

# Compare final conclusions across strategies for each patient.
conclusion_disagreement = 0
for i in range(len(PATIENT_POPULATION)):
    conclusions = {
        population_strategy_results[s][i]["conclusion"] for s in strategy_names
    }
    if len(conclusions) > 1:
        conclusion_disagreement += 1

print(f"Patients for whom at least one strategy reaches a different conclusion: "
      f"{conclusion_disagreement} / {len(PATIENT_POPULATION)}")

# Demonstration-patient strategy comparison
demo_strategy_table = pd.DataFrame([
    {"Strategy":r["strategy"], "Questions asked":len(r["questions"]),
     "Question sequence":" → ".join(r["questions"]),
     "Conclusion":r["conclusion"], "Leading CF":r["confidence"]}
    for r in consultation_results
])
display(demo_strategy_table)

### Exercise 2 — Interpretation

- The demonstration-patient table identifies which strategy asks the fewest questions for that particular patient and whether it reaches the same conclusion as the other strategies.
- The population summary shows whether that strategy is also best on average. A strategy that is best for one patient need not minimize questions for the entire population.
- Goal-directed questioning can reduce unnecessary questions when a leading diagnosis emerges early, but it depends on the quality of the current leading hypothesis. Frequency ordering is a general-purpose heuristic, while a fixed intake form is predictable and may be easier to standardize.
- In a real clinic, question order should balance information value, patient burden, safety-critical red flags, and clinical workflow. This toy system does not validate clinical safety or diagnostic accuracy.

## 9. Conclusion

The experiment implements a certainty-factor expert system with eight rules and four candidate diagnoses. It represents uncertain patient reports numerically, evaluates conjunctions using the minimum operator, scales each rule contribution by its rule CF, and combines evidence into a ranked differential. A threshold prevents the system from forcing a diagnosis when evidence is weak. The consultation component compares fixed, frequency-based, and goal-directed question selection. Finally, the two exercises evaluate robustness to rule-CF perturbations and compare question burden across a synthetic patient population.

**Limitations:** The rule base and certainty values are illustrative rather than clinically validated. The model assumes a simplified set of findings, does not account for dependencies between symptoms, and does not include examination findings, patient history, test results, prevalence, or clinician judgement. It is an educational demonstration only.